# scVI integration of only the T cells, B cells, Plasma cells, and NK cells for the revision
- removing sample05 (spleen)
- removing cluster that looks like tumor doublets

In [1]:
import os
import sys
import numpy as np
import scanpy as sc
import torch
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
mpl.rcParams['ps.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

module_path = '/labs/delitto/james/functions/'
sys.path.append(module_path)
import jpascvi

In [2]:
torch.cuda.is_available()

True

In [3]:
# version control
print("seaborn:", sns.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)
print("scvi:", scvi.__version__)
scvi.settings.seed = 1234
sns.set_theme()
torch.set_float32_matmul_precision("high")

[rank: 0] Seed set to 1234


seaborn: 0.13.2
pandas: 2.2.2
numpy: 1.26.4
scanpy: 1.10.2
scvi: 1.1.6


In [4]:
# Set up input and output directories
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent
print(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / 'objects'
print(DATA_DIR)

output_dir = jpascvi.create_output_dir(PROJECT_DIR, 'lymphoid_wo5_wodoublet', change_dir=True)

/oak/stanford/groups/longaker/ULMS/revision/scRNAseq
/oak/stanford/groups/longaker/ULMS/revision/scRNAseq/objects
Created output directory /oak/stanford/groups/longaker/ULMS/revision/scRNAseq/lymphoid_wo5_wodoublet
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/scRNAseq/lymphoid_wo5_wodoublet


# Subset

In [ ]:
# reload the previously annotated raw object
path = DATA_DIR / "annotated_raw_counts.h5ad"
adata = sc.read_h5ad(path)
adata

In [ ]:
np.unique(adata.obs['celltype'])

In [ ]:
# subset lymphoid cells only
lymphoid_types = ['T_and_NK', 'B', 'Plasma']
adata = adata[adata.obs['celltype'].isin(lymphoid_types)].copy()
adata

In [ ]:
#remove sample05
adata = adata[adata.obs["sample"] != "Sample05"].copy()
adata

In [ ]:
# load the previously clustered lymphoid anndata
path = PROJECT_DIR / "lymphoid_wo5/scVI_clusteredadata.h5ad"
ann = sc.read_h5ad(path)
ann

In [ ]:
sc.pl.umap(ann, color=['leiden0_3', 'CALD1'], groups='1', size=5.0)

In [ ]:
# remove the doublet cluster from the lymphoid adata
ann = ann[ann.obs['leiden0_3'] != '1'].copy()
ann

In [ ]:
adata = adata[ann.obs.index].copy()
adata

In [ ]:
del ann

In [ ]:
counts_by_batch = adata.obs.batch.value_counts()
counts_by_batch = counts_by_batch.to_dict()
print(counts_by_batch)

batch_labels = list(counts_by_batch.keys())
counts = list(counts_by_batch.values())
plt.figure(figsize=(10, 5))
plt.bar(batch_labels, counts)

# Add labels and title
plt.xlabel('Batch')
plt.ylabel('Counts')
plt.title('Counts Per Batch in Lymphoid Cells')
plt.xticks(rotation='vertical')

# Display the plot
plt.show()
plt.savefig('counts_by_batch.png')
plt.close()

In [ ]:
counts_by_sample = adata.obs['sample'].value_counts()
counts_by_sample = counts_by_sample.to_dict()
print(counts_by_sample)

sample_labels = list(counts_by_sample.keys())
counts = list(counts_by_sample.values())
plt.figure(figsize=(10, 5))
plt.bar(sample_labels, counts)

# Add labels and title
plt.xlabel('Sample')
plt.ylabel('Counts')
plt.title('Counts Per Sample in Lymphoid Cells')
plt.xticks(rotation='vertical')

# Display the plot
plt.show()
plt.savefig('counts_by_sample.png')
plt.close()

# Prepare for scVI

In [ ]:
adata.layers["counts"] = adata.X.copy() # this layer will contain the raw counts
sc.pp.normalize_total(adata) # normalize X to the median total counts
sc.pp.log1p(adata) # logarithmize X
adata.raw = adata # full dimension normalized logtransformed raw data

In [ ]:
# Trying to make a histogram of counts before HVG selection
counts = adata.layers['counts'].sum(axis=1).tolist()
counts = [item for sublist in counts for item in sublist]
print(len(counts))
sns.histplot(counts)
plt.xlim(xmin=0)
plt.xlim(xmax=50000)
plt.xticks(rotation='vertical')
plt.xlabel('Counts per Cell')
plt.ylabel('Frequency')
plt.title('Total Counts Per Cell in Lymphoid Cells')
plt.xticks(rotation='vertical')
plt.show()
plt.savefig('counts_beforehvgs_histogram.png')
plt.close()

In [ ]:
print(f"Number of genes before filtering: {adata.n_vars}")
sc.pp.filter_genes(adata, min_cells=3)
print(f"Number of genes after filtering: {adata.n_vars}")

In [ ]:
print(f"Number of genes before HVG selection: {adata.n_vars}")
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    batch_key="batch",
    n_top_genes=2000,
    subset=True,
    layer="counts",
    span=0.8,
)
print(f"Number of genes after HVG selection: {adata.n_vars}")

In [ ]:
# Trying to make a histogram of counts after HVG selection
counts = adata.layers['counts'].sum(axis=1).tolist()
counts = [item for sublist in counts for item in sublist]
print(len(counts))
sns.histplot(counts)
plt.xlim(xmax=50000)
plt.xlabel('Counts per Cell')
plt.ylabel('Frequency')
plt.title('Total Counts Per Cell in Tumor Cells')
plt.xticks(rotation='vertical')
plt.savefig('counts_afterhvgs_histogram.png')
plt.show()
plt.close()

In [ ]:
# Some cells may have zero HVG counts - this may mess up integration and differential expression calculation by creating a division by zero
print(f"Number of cells in anndata: {adata.n_obs}")
# Make sure to use the raw counts layer
low_counts = adata[adata.layers['counts'].sum(axis=1) < 1]
print(f"Number of cells with zero HVG counts: {low_counts.n_obs}")

In [ ]:
# Find neighbors and UMAP prior to integration to get a baseline for batch effect
sc.tl.pca(adata)
sc.pp.neighbors(adata, key_added="X_pca")
sc.tl.umap(adata, min_dist=0.3, neighbors_key="X_pca")
sc.pl.umap(adata, neighbors_key="X_pca", color=["batch", "sample"], save='unintegrated.png')

# scVI integration

In [ ]:
# correcting for sample and batch
# Assumed that batch effect is primarily from the batch variable
scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="batch", categorical_covariate_keys=['sample',])
model = scvi.model.SCVI(adata)
print(model)

# Train the vae with early stopping for the default number of epochs
scvi.settings.seed = 1234
model.train(check_val_every_n_epoch=1,
            early_stopping=True,
            early_stopping_patience=20, # how many epochs of no change are tolerated
            early_stopping_monitor="elbo_validation")

# Check training
train_test_results = model.history["elbo_train"]
train_test_results["elbo_validation"] = model.history["elbo_validation"]
train_test_results.plot()
plt.savefig('elbo_plot.png')
plt.close()

In [ ]:
adata.obsm["X_scVI"] = model.get_latent_representation()
sc.pp.neighbors(adata, use_rep="X_scVI", key_added="N_scVI")
sc.tl.umap(adata, min_dist=0.3, neighbors_key="N_scVI")
adata.layers["scvi_normalized"] = model.get_normalized_expression()
# saving the model and anndata now that umap has been computed
model.save(dir_path=output_dir, prefix='scVI', overwrite=True, save_anndata=True)

# Feature plots

In [5]:
jpa_markers = jpascvi.import_markers('/labs/delitto/james/ref/jpa_sc_markers.csv', output_type='dict')
mmk_markers = jpascvi.import_markers('/labs/delitto/james/ref/mmk_sc_markers.csv', output_type='dict')
djd_markers = jpascvi.import_markers('/labs/delitto/ulms_cellbender/ref/markers_4.csv', output_type='dict')
lymphoid_markers = jpascvi.import_markers('/labs/delitto/james/ref/lymphoid.csv', output_type='dict')

{'B_cell': ['PTPRC', 'BANK1', 'MS4A1', 'CD79A', 'CD19', 'VPREB3'], 'Plasma_cell': ['CD38', 'SDC1', 'CD79A', 'MZB1', 'JCHAIN'], 'T_cell': ['CD3E', 'IL7R', 'FYN', 'CD4', 'CD8A', 'XCL1'], 'NK_cell': ['GNLY', 'FCGR3A', 'NCAM1', 'KLRB1', 'KLRF1', 'KLRD1', 'NKG7', 'GZMA', 'GZMB'], 'Monocyte': ['FCGR3A', 'CD14', 'LYZ', 'MS4A7'], 'Macrophage': ['CD163', 'CD68', 'TLR2', 'C1QA', 'C1QB', 'C1QC', 'FCER1G'], 'cDC': ['ITGAX', 'ITGAM', 'CADM1', 'XCR1', 'CLEC9A', 'RAB32', 'C1orf54', 'THBD', 'CD1C', 'FCER1A', 'CLEC10A', 'CD1E', 'VCAN', 'CD14', 'FCGR2B', 'STAB1', 'CD226', 'LSP1'], 'pDC': ['CLEC4C', 'TCF4', 'IL3RA', 'RHEX', 'TLR7', 'TLR9', 'IRF8', 'LILRA4', 'BCL11A', 'NRP1'], 'Neutrophil': ['CSF3R', 'NAMPT', 'CXCL8', 'MNDA'], 'Mast_cell': ['TPSB2', 'TPSAB1', 'MS4A2', 'CTSG'], 'Fibroblast': ['COL1A1', 'COL1A2', 'PDPN', 'LUM', 'PODN', 'DCN', 'LAMA2', 'CDH19', 'BNC2', 'ZFPM2', 'CFD', 'MGP', 'ADAM12', 'GPC3', 'SFRP2', 'MMP3', 'VIM', 'ALDH1A1', 'THY1', 'FN1', 'OGN', 'COL3A1'], 'Pericyte': ['RGS5', 'MCAM', 'PD

In [6]:
sc._settings.ScanpyConfig.figdir = output_dir
sc._settings.ScanpyConfig.autoshow = False
sc._settings.ScanpyConfig.autosave = True

In [ ]:
jpascvi.featureplot(adata, mmk_markers, neighbors_key="N_scVI")
jpascvi.featureplot(adata, jpa_markers, neighbors_key="N_scVI")
jpascvi.featureplot(adata, lymphoid_markers, neighbors_key="N_scVI")

In [ ]:
# QC umap
sc.pl.umap(adata, 
           neighbors_key="N_scVI", 
           color=['n_genes_by_counts', 'log1p_n_genes_by_counts', 
                   'total_counts', 'log1p_total_counts', 'n_counts', 
                   'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 
                   'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 
                   'doublet_score', 'doublet',], 
           frameon=False, ncols=4, save='qc_umap.png',)

sc.pl.umap(adata, neighbors_key='N_scVI', color='batch', frameon=False, save='batch.png')
sc.pl.umap(adata, neighbors_key='N_scVI', color='sample', frameon=False, save='sample.png')

# Main loop: clustering

In [ ]:
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
for resolution in resolutions:
    print("Clustering with resolution " + str(resolution))
    str_res = str(resolution).replace('.', '_')
    leiden_key = "leiden" + str_res
    sc.tl.leiden(adata, neighbors_key="N_scVI", key_added=leiden_key, resolution=resolution, flavor="igraph", n_iterations=2)
    jpascvi.plot_umap(adata, resolution, neighbors_key="N_scVI")
    jpascvi.scvi_degs(adata, model, resolution, djd_markers, rep_key="X_scVI", norm_layer="scvi_normalized")
    jpascvi.sc_degs(adata, resolution, use_rep='X_scVI', plots=['dotplot'])

    # dotplot
    sc.pl.dotplot(adata, lymphoid_markers, groupby=leiden_key, dendrogram=False,
                  swap_axes=True, use_raw=True, standard_scale="var", save=f'lymphoid_dp_{str_res}.png')
    # heatmap
    sc.pl.heatmap(adata, lymphoid_markers, groupby=leiden_key, 
                  standard_scale="var", dendrogram=False, save=f'lymphoid_hm_{str_res}.png')
    # matrix plot
    sc.pl.matrixplot(adata, lymphoid_markers, groupby=leiden_key, standard_scale="var", save=f'lymphoid_mp_{str_res}.png')

# Save adata with umap and leiden clustering
model.save(dir_path=output_dir, prefix='scVI_clustered', overwrite=True, save_anndata=True)

In [ ]:
jpascvi.cluster_stats(adata, resolutions)

# Adding resolutions

In [7]:
adata = sc.read_h5ad('scVI_clusteredadata.h5ad')
print(adata)
model = scvi.model.SCVI.load(output_dir, prefix='scVI_clustered', adata=adata)
print(model)

AnnData object with n_obs × n_vars = 30037 × 2000
    obs: 'batch', 'sample', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'doublet', 'doublet_score', 'celltype', '_scvi_batch', '_scvi_labels', 'leiden0_1', '_scvi_raw_norm_scaling', 'leiden0_2', 'leiden0_3', 'leiden0_4', 'leiden0_5', 'leiden0_6', 'leiden0_7', 'leiden0_8', 'leiden0_9', 'leiden1_0'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'N_scVI', 'X_pca', '_scvi_manager_uuid', '_scvi_uuid', 'batch_colors', 'dendrogram_leiden0_1', 'dendrogram_leiden0_2', 'dendrogram_leiden0_3', 'dendrogram_leiden0_4', 'dendrogram_leiden0_5', 'dendr

/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scvi/model/base/_utils.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path, map_

SCVI model with the following parameters: 
n_hidden: 128, n_latent: 10, n_layers: 1, dropout_rate: 0.1, dispersion: gene, gene_likelihood: zinb, 
latent_distribution: normal.
Training status: Trained
Model's adata is minified?: False

In [8]:
resolutions = [1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
for resolution in resolutions:
    print("Clustering with resolution " + str(resolution))
    str_res = str(resolution).replace('.', '_')
    leiden_key = "leiden" + str_res
    sc.tl.leiden(adata, neighbors_key="N_scVI", key_added=leiden_key, resolution=resolution, flavor="igraph", n_iterations=2)
    jpascvi.plot_umap(adata, resolution, neighbors_key="N_scVI")
    jpascvi.scvi_degs(adata, model, resolution, djd_markers, rep_key="X_scVI", norm_layer="scvi_normalized")
    jpascvi.sc_degs(adata, resolution, use_rep='X_scVI', plots=['dotplot'])

    # dotplot
    sc.pl.dotplot(adata, lymphoid_markers, groupby=leiden_key, dendrogram=False,
                  swap_axes=True, use_raw=True, standard_scale="var", save=f'lymphoid_dp_{str_res}.png')
    # heatmap
    sc.pl.heatmap(adata, lymphoid_markers, groupby=leiden_key, 
                  standard_scale="var", dendrogram=False, save=f'lymphoid_hm_{str_res}.png')
    # matrix plot
    sc.pl.matrixplot(adata, lymphoid_markers, groupby=leiden_key, standard_scale="var", save=f'lymphoid_mp_{str_res}.png')

# Save adata with umap and leiden clustering
model.save(dir_path=output_dir, prefix='scVI_clustered', overwrite=True, save_anndata=True)

Clustering with resolution 1.1
DE...: 100%|██████████| 20/20 [00:50<00:00,  2.51s/it]
Calculating scanpy DEGs for resolution 1.1


/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


Clustering with resolution 1.2
DE...: 100%|██████████| 22/22 [00:58<00:00,  2.66s/it]
Calculating scanpy DEGs for resolution 1.2


/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:440: PerformanceWa

Clustering with resolution 1.3
DE...: 100%|██████████| 24/24 [01:02<00:00,  2.62s/it]
Calculating scanpy DEGs for resolution 1.3


...

/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:440: PerformanceWa

Clustering with resolution 1.9
DE...: 100%|██████████| 32/32 [01:22<00:00,  2.58s/it]
Calculating scanpy DEGs for resolution 1.9


/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:440: PerformanceWa

Clustering with resolution 2.0
DE...: 100%|██████████| 32/32 [01:23<00:00,  2.62s/it]
Calculating scanpy DEGs for resolution 2.0


/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:440: PerformanceWa

In [9]:
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
               1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
jpascvi.cluster_stats(adata, resolutions)

Calculating silhouette scores
leiden0_1 0.06289101
leiden0_2 0.09177814
leiden0_3 0.092294514
leiden0_4 0.091321126
leiden0_5 0.058564845
leiden0_6 0.07190483
leiden0_7 0.058016323
leiden0_8 0.0474408
leiden0_9 0.038793053
leiden1_0 0.0325867
leiden1_1 0.03400533
leiden1_2 0.024013946
leiden1_3 0.027415898
leiden1_4 0.006207044
leiden1_5 0.006351513
leiden1_6 0.010332207
leiden1_7 0.010811011
leiden1_8 0.016455803
leiden1_9 0.011298047
leiden2_0 0.008684146
Calculating Calinski-Harabasz scores
leiden0_1 748.088041022881
leiden0_2 1489.4876676625117
leiden0_3 1450.6107680413722
leiden0_4 1436.741233422347
leiden0_5 1218.735032050288
leiden0_6 1229.5404734797537
leiden0_7 1181.999006813625
leiden0_8 1034.121117422259
leiden0_9 947.5850948335948
leiden1_0 853.3540754426188
leiden1_1 843.4028929579979
leiden1_2 757.0761831532994
leiden1_3 750.9254981142043
leiden1_4 676.051470955805
leiden1_5 656.4057751991286
leiden1_6 629.0172822454753
leiden1_7 630.2781498299116
leiden1_8 582.7733040325

# Cell cycle scoring
- https://nbviewer.org/github/theislab/scanpy_usage/blob/master/180209_cell_cycle/cell_cycle.ipynb
- https://satijalab.org/seurat/articles/cell_cycle_vignette.html#assign-cell-cycle-scores

In [ ]:
# reload the previously clustered and umapped object
ad_path = output_dir / 'scVI_clusteredadata.h5ad'
adata = sc.read_h5ad(ad_path)
print(adata)
# need all the genes, not just HVGs
rawdata = adata.raw.to_adata()
print(rawdata)
del adata

In [ ]:
# get the cell cycle genes from https://www.science.org/doi/10.1126/science.aad0501
# download from https://www.dropbox.com/s/3dby3bjsaf5arrw/cell_cycle_vignette_files.zip?dl=1
cell_cycle_genes = [x.strip() for x in open('/labs/delitto/james/ref/regev_lab_cell_cycle_genes.txt')]
s_genes = cell_cycle_genes[:43]
s_genes = [x for x in s_genes if x in rawdata.var_names]
g2m_genes = cell_cycle_genes[43:]
g2m_genes = [x for x in g2m_genes if x in rawdata.var_names]
cell_cycle_genes = [x for x in cell_cycle_genes if x in rawdata.var_names]

In [ ]:
# Data is already log-transformed, but we still need to scale data to unit variance and zero mean in order to score genes
sc.pp.scale(rawdata)

In [ ]:
sc.tl.score_genes_cell_cycle(rawdata, s_genes=s_genes, g2m_genes=g2m_genes)
print(rawdata)

In [ ]:
sc.pl.umap(rawdata, neighbors_key='N_scVI', color=['leiden0_2', 'phase', 'S_score', 'G2M_score'], ncols=2, save='cell_cycle.png')

In [ ]:
del rawdata

# Automated annotation with celltypist

In [10]:
import celltypist as ct
from celltypist import models
print("celltypist:", ct.__version__)

celltypist: 1.6.3


celltypist expects anndata normalized to 1e4 and log-transformed with all the genes, and I didn't do that, so let's recreate the anndata

In [11]:
# reload the previously clustered and umapped object
ad_path = output_dir / 'scVI_clusteredadata.h5ad'
adata = sc.read_h5ad(ad_path)
print(adata)
print()

# load the rawdata and filter for lymphoid cells only
ad_path = DATA_DIR / 'annotated_raw_counts.h5ad'
rawdata = sc.read_h5ad(ad_path)
print(rawdata)
print()

# subset the raw counts and transfer scVI embedding
rawdata = rawdata[adata.obs.index].copy()
rawdata.uns['N_scVI'] = adata.uns['N_scVI']
rawdata.obsm['X_scVI'] = adata.obsm['X_scVI']
rawdata.obsm['X_umap'] = adata.obsm['X_umap']
rawdata.obsp['N_scVI_connectivities'] = adata.obsp['N_scVI_connectivities']
rawdata.obsp['N_scVI_distances'] = adata.obsp['N_scVI_distances']
leiden_cols = adata.obs.columns[adata.obs.columns.str.startswith('leiden')]
rawdata.obs[leiden_cols] = adata.obs.loc[rawdata.obs.index, leiden_cols]
del adata
adata = rawdata
del rawdata
print(adata)

AnnData object with n_obs × n_vars = 30037 × 2000
    obs: 'batch', 'sample', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'doublet', 'doublet_score', 'celltype', '_scvi_batch', '_scvi_labels', 'leiden0_1', '_scvi_raw_norm_scaling', 'leiden0_2', 'leiden0_3', 'leiden0_4', 'leiden0_5', 'leiden0_6', 'leiden0_7', 'leiden0_8', 'leiden0_9', 'leiden1_0', 'leiden1_1', 'leiden1_2', 'leiden1_3', 'leiden1_4', 'leiden1_5', 'leiden1_6', 'leiden1_7', 'leiden1_8', 'leiden1_9', 'leiden2_0'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'N_scVI', 'X_pca', '_scvi_manager_uuid', '_scvi_uuid', 'batch_color

In [ ]:
adata.layers["counts"] = adata.X.copy() # this layer will contain the raw counts
sc.pp.normalize_total(adata, target_sum=1e4) # normalize X to the median total counts
sc.pp.log1p(adata) # logarithmize X
adata.raw = adata # full dimension normalized logtransformed raw data

In [ ]:
model = models.Model.load(model='Immune_All_Low.pkl')
model

In [ ]:
model.cell_types

In [ ]:
leiden_key = 'leiden1_5'
predictions = ct.annotate(adata, model=model, majority_voting=True, over_clustering=leiden_key)

In [ ]:
predictions.predicted_labels

In [ ]:
adata = predictions.to_adata()
adata.obs

In [ ]:
sc.pl.umap(adata, ncols=1,
           neighbors_key="N_scVI", 
           color=['majority_voting'], 
           frameon=False, save='celltypist1_0_mv.png',)